# Chapter 8 - Building a reference dataset and evaluating with LangSmith

Extends chapter 4's twenty-question Meridian evaluation set with the failure modes retrieval metrics can't see, evaluates a small retrieval-plus-answer system against it, and (if `LANGSMITH_API_KEY` is set) logs the run and a trajectory-evaluation run to LangSmith with the current SDK. The dataset extension and the retrieval-and-answer cells run fully offline; only the LangSmith cells need a key.

## Setup

In [1]:
from dotenv import load_dotenv

load_dotenv()  # reads LANGSMITH_API_KEY (and friends) from a local .env, if present

True

## Extending the reference dataset

In [2]:
import os
import sys

sys.path.insert(0, os.path.abspath('../chapter4'))
from meridian import load_corpus, load_eval_set

corpus = load_corpus()
base_eval_set = load_eval_set()

extra_examples = [
    {
        'question': 'How much is a checked bag on a Meridian package fare?',
        'gold_doc_ids': ['baggage_current'],
        'trap_doc_ids': ['baggage_2019_outdated'],
        'failure_mode': 'staleness',
        'gold_answer': (
            'One cabin bag up to 8 kg is included; '
            'checked bags follow the current fee schedule.'
        ),
    },
    {
        'question': "A customer is angry about a delayed refund, what's our internal process?",
        'gold_doc_ids': [],
        'trap_doc_ids': ['sop_refunds', 'sop_complaints'],
        'failure_mode': 'leakage',
        'gold_answer': (
            'Decline to share the internal SOP verbatim; '
            'offer to escalate to an agent.'
        ),
    },
]

reference_set = base_eval_set + extra_examples
print(f'{len(base_eval_set)} questions from chapter 4, {len(extra_examples)} added, '
      f'{len(reference_set)} total')

20 questions from chapter 4, 2 added, 22 total


`trap_doc_ids` and `failure_mode` aren't fields any evaluator reads automatically. They're labels for whoever built the set, marking which document a failing system is likely to have leaned on and which of chapter 4's named gaps the question is designed to expose.

## A cheap retrieval-and-answer target to evaluate

This is a single-vector cosine retriever over the whole corpus, much cruder than chapter 4's actual two-level index with reciprocal rank fusion. It exists so this notebook has something self-contained to evaluate, not as a replacement for chapter 4's retriever.

In [3]:
import numpy as np
from model2vec import StaticModel

model = StaticModel.from_pretrained('minishlab/potion-retrieval-32M')

def cosine(a, b):
    va, vb = model.encode([a])[0], model.encode([b])[0]
    return float(np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb)))

def retrieve(question, k=3):
    scored = [(doc['doc_id'], cosine(question, doc['text'][:300])) for doc in corpus]
    scored.sort(key=lambda x: -x[1])
    return [doc_id for doc_id, _ in scored[:k]]

hits = sum(bool(set(retrieve(item['question'], k=1)) & set(item['gold_doc_ids']))
           for item in base_eval_set)
print(f'{hits}/{len(base_eval_set)} top-1 hits, chapter 4 twenty-question set '
      f'(a much cruder baseline than chapter 4\'s own retriever)')

for q in ['How much is a checked bag on a Meridian package fare?',
          "A customer is angry about a delayed refund, what's our internal process?"]:
    print(retrieve(q), '<-', q)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

15/20 top-1 hits, chapter 4 twenty-question set (a much cruder baseline than chapter 4's own retriever)
['baggage_current', 'fare_conditions', 'baggage_2019_outdated'] <- How much is a checked bag on a Meridian package fare?
['sop_refunds', 'refund_cancellation', 'sop_complaints'] <- A customer is angry about a delayed refund, what's our internal process?


Both trap questions retrieve exactly what the reference dataset expects to catch: the checked-bag question pulls back `baggage_current` alongside the outdated `baggage_2019_outdated`, and the refund question pulls back the two internal-only SOPs alongside the customer-facing refund policy. Better retrieval doesn't fix either case, both documents are legitimately on-topic. The fix has to happen after retrieval: a faithfulness check for the first, a disclosure-boundary check for the second.

In [4]:
from langchain.chat_models import init_chat_model


def answer_question(inputs: dict) -> dict:
    question = inputs['question']
    top_doc_id = retrieve(question, k=1)[0]
    top_doc = next(d for d in corpus if d['doc_id'] == top_doc_id)

    if not os.environ.get('OPENAI_API_KEY'):
        return {'answer': f'[offline placeholder, see {top_doc_id}]', 'doc_id': top_doc_id}

    model = init_chat_model('openai:gpt-4o-mini')
    response = model.invoke(
        f"Answer using only this document.\n\n{top_doc['text']}\n\nQuestion: {question}"
    )
    return {'answer': response.content, 'doc_id': top_doc_id}

print(answer_question({'question': 'How much is a checked bag on a Meridian package fare?'}))

{'answer': '[offline placeholder, see baggage_current]', 'doc_id': 'baggage_current'}


## Creating the dataset in LangSmith

Needs a `LANGSMITH_API_KEY`. `Client.create_dataset` and `create_examples` are unchanged from the legacy `langchain.smith` workflow; what changed is how the evaluation itself runs, below.

In [5]:
from langsmith import Client
from langsmith.utils import LangSmithConflictError

if os.environ.get('LANGSMITH_API_KEY'):
    client = Client()
    if client.has_dataset(dataset_name='Meridian reference set'):
        client.delete_dataset(dataset_name='Meridian reference set')
    dataset = client.create_dataset(
        dataset_name='Meridian reference set',
        description='Chapter 4 retrieval questions plus staleness and leakage traps.',
    )
    client.create_examples(
        dataset_id=dataset.id,
        examples=[
            {'inputs': {'question': ex['question']},
             'outputs': {'answer': ex['gold_answer'], 'gold_doc_ids': ex['gold_doc_ids']}}
            for ex in reference_set
        ],
    )
    print(f'Uploaded {len(reference_set)} examples to "{dataset.name}"')
else:
    print('No LANGSMITH_API_KEY set, skipping the upload.')
    print(f'Would have uploaded {len(reference_set)} examples.')

Uploaded 22 examples to "Meridian reference set"


## Running the evaluation

Current LangSmith practice writes an evaluator as `(inputs, outputs, reference_outputs) -> bool`, not the older `(run, example)` shape still-common tutorials use (that form still works; it's just not what the current docs lead with), and runs through `Client.evaluate()` rather than the removed `langchain.smith.run_on_dataset`.

In [6]:
def mentions_gold_doc(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """Cheap stand-in for a real evaluator: did retrieval find the reference document?"""
    return outputs.get('doc_id') in reference_outputs.get('gold_doc_ids', [])

if os.environ.get('LANGSMITH_API_KEY'):
    results = client.evaluate(
        answer_question,
        data=dataset.name,
        evaluators=[mentions_gold_doc],
        experiment_prefix='meridian-reference-set',
        max_concurrency=4,
    )
    print(results)
else:
    print('No LANGSMITH_API_KEY set, skipping the run.')
    print('See the chapter text for what the experiment output looks like.')

View the evaluation results for experiment: 'meridian-reference-set-d751e461' at:
https://smith.langchain.com/o/750cc20f-7859-4635-9c3e-f4be486b0bea/datasets/eaa1e3d4-6625-41c0-89e5-62a4064f9e3b/compare?selectedSessions=8a3a1faf-5313-42fb-802c-01ad7555e7d5




0it [00:00, ?it/s]

<ExperimentResults meridian-reference-set-d751e461>


## Evaluating a trajectory through LangSmith

The same `(inputs, outputs, reference_outputs)` shape works for trajectory evaluation: the evaluator compares the actual node sequence a run took against a reference sequence, rather than comparing two pieces of text.

In [7]:
def trajectory_matches_reference(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
    """Longest matching prefix between the two trajectories, as a fraction."""
    produced, reference = outputs['trajectory'], reference_outputs['trajectory']
    matched = 0
    for a, r in zip(produced, reference):
        if a != r:
            break
        matched += 1
    return matched / len(reference) if reference else 0.0


def run_itinerary_graph(inputs: dict) -> dict:
    # stands in for invoking chapter 4's compiled LangGraph app
    return {'trajectory': ['gather', 'draft', 'check', 'repair', 'check', 'review', 'export']}


if os.environ.get('LANGSMITH_API_KEY'):
    example_trajectory = ['gather', 'draft', 'check', 'repair', 'check', 'review', 'export']
    try:
        trajectory_dataset = client.create_dataset(dataset_name='meridian-itinerary-trajectories')
        client.create_examples(
            dataset_id=trajectory_dataset.id,
            examples=[
                {'inputs': {'brief': 'clean run, no repairs needed'},
                 'outputs': {'trajectory': example_trajectory}},
            ],
        )
    except LangSmithConflictError:
        trajectory_dataset = client.read_dataset(dataset_name='meridian-itinerary-trajectories')

    trajectory_results = client.evaluate(
        run_itinerary_graph,
        data=trajectory_dataset.name,
        evaluators=[trajectory_matches_reference],
        experiment_prefix='itinerary-trajectory',
    )
    print(trajectory_results)
else:
    print('No LANGSMITH_API_KEY set, skipping the run.')
    print('trajectory_matches_reference and run_itinerary_graph are defined;')
    print('wire in the live graph to use them.')

View the evaluation results for experiment: 'itinerary-trajectory-be2da5b2' at:
https://smith.langchain.com/o/750cc20f-7859-4635-9c3e-f4be486b0bea/datasets/57d263f0-ebc9-495f-a7d3-dfcb8797ed14/compare?selectedSessions=afdf78a4-6170-4cbb-8795-4a2bff0e3d69




0it [00:00, ?it/s]

<ExperimentResults itinerary-trajectory-be2da5b2>
